# Informe Final Integral: Análisis de la Industria de Videojuegos con Datos de IGDB

## Resumen Ejecutivo

Este documento consolida el ciclo de vida completo del proyecto de Ciencia de Datos, desde la ingestión de datos crudos hasta el modelado predictivo. El objetivo principal es transformar datos técnicos en decisiones estratégicas para desarrolladores de videojuegos independientes (“indies”), identificando los factores clave que separan a un juego exitoso del resto.

Originalmente se evaluó trabajar con múltiples fuentes (RAWG, IGDB y Steam/SteamCharts), pero problemas de descontinuación e inestabilidad de algunas APIs llevaron a concentrar el proyecto en una única fuente principal: **la API oficial de IGDB**. Esto permitió construir un pipeline reproducible, completo y alineado con las buenas prácticas de ingeniería de datos.

---

## 1. Fase de Extracción de Datos (ETL – Extraction)

**Notebook de referencia:** `notebooks/api/igdb_api_request.ipynb`  
**Script de apoyo:** `codigos/descarga.py`

En esta etapa se definió la estrategia para obtener la materia prima del proyecto.

### 1.1 Selección de la fuente de datos

Se optó por utilizar la **API oficial de IGDB** como fuente de datos **secundaria y externa**.

- **Justificación técnica:**
  - A diferencia del *web scraping*, que extrae datos no estructurados (HTML) muy sensibles a cambios visuales, la API entrega datos **semi-estructurados (JSON)** con:
    - esquemas documentados,
    - relaciones explícitas entre entidades (juegos, compañías, plataformas, géneros),
    - y soporte para filtros y paginación.
- **Justificación ética y de buenas prácticas:**
  - El uso de una API documentada respeta los términos de servicio del proveedor,
  - evita saturar servidores con peticiones no controladas,
  - y se alinea con el enfoque profesional de ingeniería de datos.

### 1.2 Metodología de extracción

Se implementó un proceso de descarga iterativa (paginación) desde distintos endpoints de IGDB.

- **Desafío:** La base de datos contiene cientos de miles de registros; una sola petición gigante habría excedido límites de tiempo, memoria y *rate limits*.
- **Solución técnica:**
  - Se diseñó un bucle que descarga lotes de tamaño fijo (`limit = 500`) usando un puntero de desplazamiento (`offset`).
  - Para cada endpoint (`games`, `genres`, `platforms`, `companies`, `involved_companies`) se repite el proceso hasta agotar resultados.
- **Control de flujo y *Rate Limits*:**
  - Se incorporó un retardo artificial (`time.sleep`) entre peticiones.
  - **Motivación:** respetar el límite de ~4 requests por segundo, evitar bloqueos de IP y garantizar la reproducibilidad del proceso.
- **Resultado:** Archivos CSV crudos guardados en `data/raw/`:
  - `juegos_raw.csv`
  - `igdb_generos.csv`
  - `igdb_plataformas.csv`
  - `igdb_empresas.csv`
  - `igdb_involved_companies.csv`

---

## 2. Fase de Limpieza y Transformación (Data Wrangling)

**Notebooks de referencia:**  
- `00_limpieza_raw_g.ipynb` (juegos)  
- `01_limpieza_auxiliares_igdb.ipynb` (tablas auxiliares)

Aquí abordamos la “T” de ETL, transformando datos crudos en un formato analítico confiable.

### 2.1 Manejo de tipos de datos complejos

- **Problema:**
  - Al exportar JSON a CSV, estructuras de listas como `[1, 2, 3]` se convirtieron en cadenas de texto (ej: `"[1, 2, 3]"`).
  - Esto afectaba campos clave como `genres`, `platforms` o `game_modes`.
- **Solución:**
  - Usar `ast.literal_eval` para reconstruir listas de enteros desde el string.
- **Justificación:**
  - `literal_eval` interpreta literales de manera segura (a diferencia de `eval`, que ejecuta código arbitrario).
  - Esto permitió luego crear tablas puente **juego–género** y **juego–plataforma** para el análisis y el modelo.

### 2.2 Definición de ventana temporal y calidad de datos

- **Decisión:** filtrar juegos lanzados exclusivamente entre **2000 y 2025**.
- **Justificación:**
  - El *data profiling* mostró que los registros anteriores al 2000 tenían un porcentaje muy alto de valores nulos en métricas críticas (ratings, reseñas).
  - Además, reflejan un mercado con plataformas, modelos de negocio y dinámicas muy distintas al contexto actual.
- **Consecuencia:**
  - Se concentra el análisis en el mercado moderno, sacrificando alcance histórico a cambio de mayor calidad y coherencia.

### 2.3 Lógica de negocio para duplicados

- **Hallazgo:**
  - Un mismo título aparece múltiples veces por relanzamientos, versiones “GOTY”, ports, versiones “enhanced”, etc.
- **Tratamiento aplicado:**
  - Agrupar por **nombre normalizado de juego** (y año cuando corresponde).
  - Mantener la fila con **mayor `total_rating_count`**.
- **Motivación:**
  - Esa versión se interpreta como la edición principal o más visible comercialmente.
  - Se evita borrar información relevante al azar y se conserva la señal con mayor interacción de usuarios.

### 2.4 Limpieza de tablas auxiliares

En `01_limpieza_auxiliares_igdb.ipynb` se limpiaron:

- `igdb_generos.csv` → `igdb_generos_limpio.csv`
- `igdb_plataformas.csv` → `igdb_plataformas_limpio.csv`
- `igdb_empresas.csv` → `igdb_empresas_limpio.csv`
- `igdb_involved_companies.csv` → `igdb_involved_companies_limpio.csv`

Pasos principales:

- Normalización de nombres e ids.
- Eliminación de columnas irrelevantes o casi vacías.
- Preparación de tablas puente:
  - `df_juegos_generos` (relación juego–género),
  - `df_juegos_plat` (relación juego–plataforma),
  - tabla de publishers por juego a partir de `involved_companies`.

---

## 3. Fase de Análisis Exploratorio de Datos (EDA)

**Notebook de referencia:** `02_eda_igdb.ipynb`

En esta fase se revisaron distribuciones, correlaciones y se respondieron las preguntas de negocio planteadas en la propuesta.

### 3.1 Datos faltantes y mecanismo MNAR

Se observó que `total_rating` y `total_rating_count` tenían un porcentaje muy alto de valores nulos (más del 80 %).

- **Conclusión teórica:**
  - No se trata de datos faltantes completamente al azar (MCAR).
  - Es un mecanismo **MNAR (Missing Not At Random)**:
    - un juego no tiene rating porque simplemente no ha recibido reseñas,
    - y eso suele estar asociado a baja visibilidad o poco éxito comercial.
- **Decisión metodológica:**
  - Para definir y analizar “éxito” se trabajó solo con el subconjunto de juegos que **sí tienen** `total_rating` y `total_rating_count`.
  - Los juegos sin estas métricas no se marcan como “fracaso”, sino como **éxito desconocido**.

### 3.2 Ingeniería de la variable objetivo (Y)

Para habilitar el aprendizaje supervisado se necesitaba una definición concreta de “juego exitoso”.

- **Definición adoptada (texto plano):**

  `Exitoso = (total_rating ≥ 75) AND (total_rating_count ≥ 10)`

- **Justificación:**
  - `total_rating ≥ 75`:
    - umbral razonable para un “buen juego” en una escala de 0–100.
  - `total_rating_count ≥ 10`:
    - evita falsos positivos por uno o dos votos muy altos,
    - requiere cierto nivel mínimo de atención comunitaria.
- **Manejo de casos:**
  - Si `total_rating` y `total_rating_count` están presentes:
    - se evalúa la condición anterior → `exitoso` es `True` o `False`.
  - Si faltan una o ambas métricas:
    - `exitoso = NaN` (no evaluable).
- **Impacto:**
  - Solo ~13–16 % de los juegos evaluables son exitosos → dataset **desbalanceado**.

### 3.3 Tipo de estudio (AAA vs No AAA / Sin publisher)

A partir de `involved_companies` y `empresas` se identificaron publishers para cada juego.

- Se definió una lista curada de publishers **AAA**: Electronic Arts, Ubisoft, Nintendo, Sony, Microsoft, Square Enix, Rockstar, etc.
- Con esto se creó la variable `tipo_estudio`:
  - `AAA`: al menos un publisher AAA.
  - `No AAA`: tiene publisher, pero ninguno de la lista AAA.
  - `Sin publisher`: no se registra publisher (proxy de estudio muy pequeño o autopublicado).
- **Supuesto:** “No AAA” + “Sin publisher” se usan como aproximación al mundo indie, ya que IGDB no etiqueta explícitamente “indie”.

### 3.4 Agrupación de plataformas

Para simplificar el análisis se agruparon plataformas en 5 macro-categorías:

- `PC`, `Consola`, `Portátil`, `Móvil`, `Otras`.

La asignación se hizo según substrings del nombre (“Windows”, “PlayStation”, “Android”, etc.). Es un criterio heurístico y puede contener casos borde, pero permite comparar familias de plataformas de manera útil.

---

## 3.5 Conclusiones visuales y respuestas a preguntas de negocio

A continuación se presentan los hallazgos asociados a cada pregunta de investigación, apoyados en gráficos y tablas.

### A. ¿Qué géneros dominan el mercado? (Cantidad vs. calidad)

Al cruzar año de lanzamiento, género y tasa de éxito se obtiene el siguiente mapa de calor:

![Mapa de Calor: Éxito por Género y Año](img/mapa_calorr.png)

**Hallazgos:**

- En volumen, los géneros **Indie**, **Action**, **Adventure** y **Platform** concentran la mayoría de lanzamientos recientes.
- Mirando la **proporción de juegos exitosos**:
  - el género “Indie” está **muy saturado**: muchísimos juegos, pocos que alcanzan el estándar de rating + reseñas;
  - géneros como **RPG** y **Strategy** tienen menos juegos, pero una **tasa de éxito relativamente alta y estable** en el tiempo.

**Interpretación para indies:**

Competir bajo etiquetas genéricas y saturadas (“Indie”, “Casual”) es más difícil. Apostar por géneros con comunidades exigentes pero fieles (RPG, Strategy, Simulation) puede mejorar las probabilidades de éxito relativo, aunque implique desarrollos más complejos.

---

### B. La brecha AAA vs. Indie

Para entender las diferencias estructurales se compararon distribuciones de ratings y reseñas según el tipo de estudio:

![Gráfico de Barras: Comparación AAA vs No AAA](img/aaa_noaaa.png)

**Hallazgos cuantitativos:**

- La probabilidad de éxito se ve fuertemente afectada por el tipo de estudio:
  - Juegos con publisher AAA alcanzan alrededor de **un 30–36 %** de éxito.
  - Juegos `No AAA` o `Sin publisher` apenas llegan a **2–4 %**.
- La **visibilidad**, medida como número de reseñas, es el factor crítico:
  - la mediana de `total_rating_count` en juegos AAA es muy superior a la de los indies;
  - muchos indies nunca alcanzan las 10 reseñas mínimas del criterio de éxito, incluso con buenos ratings.

**Conclusión:**

Más que la calidad técnica, el **músculo de marketing y distribución** asociado a un publisher AAA determina si un juego entra o no al radar de las tiendas digitales.

---

### C. Estilos de juego y comunidad

**Pregunta:** ¿Cómo influyen género y “estilo” de juego en la visibilidad (cantidad de reseñas)?

- Juegos de consumo rápido (Arcade, Party, ciertos Casual) tienden a acumular pocas reseñas: ciclos de juego cortos y menos incentivo a dejar opinión.
- Juegos con **profundidad** (RPG, Strategy, Tactical, Simulation) fomentan comunidades activas que:
  - escriben reseñas,
  - crean guías,
  - y mantienen el interés a largo plazo.

**Interpretación:**

Si la estrategia de un estudio indie es posicionarse vía comunidad y *word of mouth*, conviene diseñar experiencias con suficiente **profundidad y rejugabilidad** para que la gente quiera hablar del juego más allá del primer día.

---

### D. Plataformas: ¿Dónde debe lanzar un indie?

Filtrando exclusivamente juegos `No AAA` / `Sin publisher` se analizó qué categorías de plataforma entregan mejores tasas de éxito:

![Gráfico de Barras: Éxito por Plataforma](img/exito_plat.png)

**Hallazgos:**

- **PC y Consolas** muestran una **tasa de éxito considerablemente mayor** que plataformas móviles.
- En móviles, el problema de descubribilidad es extremo:
  - enorme cantidad de títulos,
  - menor cultura de dejar reseñas detalladas.

**Recomendación:**

Si el objetivo es lograr prestigio crítico y visibilidad basada en reseñas, se recomienda priorizar **PC y/o Consola** por sobre una dependencia exclusiva de plataformas móviles.

---

### E. Estacionalidad: ventanas de lanzamiento

Se analizó la proporción de éxito según el mes de lanzamiento para evaluar si existen “mejores momentos” para publicar:

![Gráfico de Barras: Éxito por Mes](img/exito_mes.png)

**Hallazgos:**

- **Enero (~8 %)** aparece como el peor mes para indies:
  - probablemente debido a la “resaca de gastos” post-navidad.
- **Septiembre y octubre (~13 %)** se ubican entre los meses con mayor proporción de juegos exitosos.
- Hay que interpretar con cautela los meses con pocos lanzamientos, donde una o dos observaciones pueden inflar las tasas, pero la tendencia global apunta a una ventana favorable en el **tercer trimestre (Q3)**.

**Recomendación:**

Planificar el lanzamiento para el tercer trimestre del año (septiembre–octubre) y evitar enero puede otorgar un pequeño aumento en la probabilidad de éxito, aunque otros factores pesan mucho más.

---

## 4. Fase de Modelado Predictivo (Machine Learning)

**Notebook de referencia:** `03_modelo.ipynb`

En esta etapa se construyeron modelos de clasificación para estimar la probabilidad de éxito:

`P(exitoso = 1 | X)`

donde `X` representa características conocidas al momento del lanzamiento.

### 4.1 Ingeniería de características (Feature Engineering)

- Se tomó como base el dataset de juegos **evaluables** (aquellos con `exitoso` definido).
- Variables incluidas:
  - **Temporales:** año y mes de lanzamiento.
  - **Tipo de estudio:** AAA, No AAA, Sin publisher.
  - **Estructura del juego:**
    - número de géneros,
    - número de plataformas,
    - indicadores (one-hot) para los **Top 10 géneros** más frecuentes.
  - **Plataformas:**
    - indicadores para categorías `PC`, `Consola`, `Portátil`, `Móvil`, `Otras`.
- Se evitó una explosión de dimensionalidad (sparsity) limitando los géneros a los más frecuentes y agrupando plataformas.

### 4.2 Manejo del desbalance de clases

- **Diagnóstico:** solo ~13 % de los juegos evaluables son exitosos.
- Sin ajustes, un modelo trivial que predice siempre “No exitoso” alcanzaría ≈87 % de *accuracy* pero no aportaría información.
- **Medidas tomadas:**
  - uso de `class_weight="balanced"` en modelos de *scikit-learn*,
  - división entrenamiento/prueba estratificada (`stratify = y`) para mantener la proporción de clases.

### 4.3 Modelos y resultados

1. **Regresión logística**
   - Rol: modelo base (*baseline*) interpretable.
   - Confirmó que:
     - pertenecer a la categoría **“Sin publisher”** reduce fuertemente la probabilidad de éxito,
     - lanzar en **Consola** (vs Móvil) aumenta la probabilidad condicional,
     - algunos géneros específicos y ciertos años también influyen.
   - Útil para entender dirección y magnitud de efectos vía coeficientes beta.

2. **Random Forest**
   - Rol: modelo no lineal robusto, capaz de capturar interacciones entre variables.
   - Obtuvo un **ROC-AUC** cercano a ~0.78, ligeramente superior a la regresión logística.
   - El análisis de importancia de variables mostró como más influyentes:
     - tipo de estudio,
     - categoría de plataforma,
     - pertenencia a géneros top,
     - año de lanzamiento.

### 4.4 Alcance del modelo

- El modelo no pretende “adivinar el futuro” con exactitud, sino servir como **brújula**:
  - dado un concepto de juego (género, plataforma, tipo de estudio, fecha tentativa), entrega una probabilidad estimada de éxito;
  - permite comparar escenarios (ej.: mismo juego en PC vs Móvil, o con / sin publisher).
- No incorpora información de contenido (historia, arte, música), por lo que **no captura creatividad**. Solo modela la estructura del mercado según datos disponibles en IGDB.

---

## 5. Sesgos, Limitaciones y Riesgos

En coherencia con la rúbrica de integridad y análisis crítico, se explicitan las principales limitaciones del proyecto:

1. **Dependencia de una sola fuente (IGDB)**
   - No se usan métricas directas de ventas ni datos de uso en tiempo real (ej. jugadores concurrentes en Steam).
   - Los resultados están condicionados a cómo IGDB recopila, modera y expone su información.

2. **Definición de éxito basada en ratings y reseñas**
   - Puede haber juegos comercialmente exitosos con pocas reseñas, dependiendo de mercados o plataformas.
   - Nuestro criterio favorece títulos con comunidades vocales; juegos “silenciosos” quedan penalizados.

3. **Datos faltantes MNAR**
   - La ausencia de ratings está relacionada con la baja visibilidad del juego; no es un missing inocente.
   - Esto introduce un sesgo inevitable en el análisis.

4. **Heurísticas en publishers y plataformas**
   - La lista de publishers considerados AAA es manual y puede omitir empresas.
   - La clasificación de plataformas (PC, Consola, Móvil, Portátil, Otras) se basa en nombres de texto y puede tener errores puntuales.

5. **Elección de umbrales (75 puntos, 10 reseñas)**
   - Son razonables pero no únicos. Cambiarlos podría alterar la proporción de éxitos y, con ello, las métricas de los modelos.

6. **Riesgo de mal uso del modelo**
   - Si se usara como “oráculo”, un estudio podría descartar ideas creativas solo porque el modelo las marca como de baja probabilidad.
   - Esto podría reforzar tendencias conservadoras y dificultar aún más la innovación en el mercado.

---

## 6. Conclusiones Generales y Recomendaciones Estratégicas

Basándonos en la evidencia empírica recolectada y modelada, se proponen las siguientes recomendaciones para estudios de videojuegos independientes:

### A. El dilema del publisher (visibilidad vs autonomía)

Los datos son claros: la variable más fuertemente asociada al fracaso es **no tener publisher**.

- **Recomendación:** la calidad del juego por sí sola no basta. Se aconseja:
  - buscar activamente un socio de publicación, o
  - invertir de forma significativa en marketing y comunidad.
- Sin visibilidad (reseñas), un juego rara vez alcanza los umbrales de éxito definidos por las tiendas.

### B. Selección de nicho (estrategia de océano azul)

El mercado de juegos genéricamente etiquetados como “Indie” o “Arcade” es un **océano rojo**: alta competencia, bajo retorno medio.

- **Recomendación:** apuntar a nichos de alta fidelidad:
  - **Strategy**, **Simulation** y **RPG** cuentan con comunidades que generan reseñas y contenido, lo que aumenta la probabilidad de cumplir los criterios de éxito.

### C. Plataformas y ventanas de lanzamiento

- **Plataformas:**
  - Evitar depender exclusivamente de móviles si el objetivo es prestigio crítico y visibilidad basada en reseñas.
  - **PC y Consolas** ofrecen mejores probabilidades de éxito para indies.
- **Fecha de lanzamiento:**
  - Planificar el lanzamiento hacia el **tercer trimestre (Q3)**, especialmente septiembre–octubre.
  - Evitar enero, el mes con menor proporción de éxito en el histórico analizado.

---

## 7. Cierre

Este proyecto demuestra cómo la aplicación rigurosa de técnicas de Ciencia de Datos —ETL, EDA y Machine Learning— permite transformar intuiciones sueltas en **decisiones de negocio informadas y cuantitativamente respaldadas**.

Aunque el modelo no reemplaza la creatividad ni el criterio de diseño, sí aporta una perspectiva estructurada sobre qué combinaciones de género, plataforma, tipo de estudio y ventana de lanzamiento ofrecen mejores probabilidades de éxito para un desarrollo independiente en la industria actual de los videojuegos.
